In [1]:
import pandas as pd
#import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
#import seaborn as sns
#import json

In [2]:
# Read and Display Data
pop_data_url = "https://raw.githubusercontent.com/so-rn/Migros-Location-Optimizer/main/Alexandros%20-%20Population/data/OCS_POPBATLOG_COMMUNE.csv"
pop_df = pd.read_csv(pop_data_url, sep=';')
pop_df.head()

,OBJECTID,DATE_REF,NO_COMMUNE,COMMUNE,BAT_TOTAL,BAT_SANS_LOG,BATLOG_TOT,MAISON_INDIV,BAT_LOG_1_2,BAT_LOG_3_9,...,POPULATION,POP_HOMMES,POP_FEMMES,POP_CH,POP_ETR,AGE_0_19,AGE_20_64,AGE_65_PLUS,SHAPE.AREA,SHAPE.LEN
0,1,12-2025,6624,Gy,265,121,144,96,123,20,...,582,266,316,467,115,137,339,106,3.286289e+06,15240.281691
1,2,12-2025,6631,Onex,2107,795,1312,829,895,164,...,19141,9157,9984,11985,7156,4071,11385,3685,2.813933e+06,8800.054395
2,3,12-2025,6641,Troinex,1309,612,697,522,587,96,...,3365,1667,1698,2573,792,772,2047,546,3.429908e+06,9160.554904
3,20,12-2025,6639,Soral,346,149,197,120,158,37,...,934,436,498,754,180,212,513,209,2.942051e+06,12467.455094
4,21,12-2025,6627,Laconnex,395,189,206,166,196,7,...,713,358,355,610,103,149,401,163,3.831199e+06,9049.018702


In [6]:
# Total population by commune
commune_population = pop_df.groupby('COMMUNE')['POPULATION'].sum().reset_index()
# Sort the communes by population
commune_population = commune_population.sort_values(by='POPULATION', ascending=False)
# Select only top 10 communes
commune_population_top10 = commune_population.head(10)

fig = px.bar(commune_population_top10, x='COMMUNE', y='POPULATION', title='Top 10 Communes by Population (2025) in Greater Geneva Area', labels={'COMMUNE': 'Commune', 'POPULATION': 'Population'})
fig.update_yaxes(type='log')
fig.show()

In [ ]:
# EXCLUDE GENEVA
pop_df_no_geneva = pop_df[pop_df["COMMUNE"] != "Genève"].copy()

# AGE COLUMNS
age_cols = ["AGE_0_19", "AGE_20_64", "AGE_65_PLUS"]

# TOTAL POPULATION
pop_df_no_geneva["TOTAL_POP"] = pop_df_no_geneva[age_cols].sum(axis=1)

# TOP-10 COMMUNES
top10 = (pop_df_no_geneva.sort_values("TOTAL_POP", ascending=False).head(10).copy())

# Bar plot with total populatios of top-10 communes excluding Geneva
fig_excl_geneve = px.bar(top10, x='COMMUNE', y='POPULATION', title='Top 10 Communes by Population (2025) in Geneva City (Excluding Geneve Commune)', labels={'COMMUNE': 'Commune', 'POPULATION': 'Population'})
fig_excl_geneve.update_layout(title_font_size=22)
fig_excl_geneve.update_xaxes(title_font_size=20, tickfont=dict(size=16))
fig_excl_geneve.update_yaxes(title_font_size=20, tickfont=dict(size=16))
fig_excl_geneve.show()

In [ ]:
'''
AGE ANALYSIS
'''

In [13]:
# Box plot of age group distributions for top-10 communes excluding Geneva
bx_plot_df = top10.melt(
    id_vars="COMMUNE",
    value_vars=age_cols,
    var_name="Age_Group",
    value_name="Population"
)

# cleaner labels
bx_plot_df["Age_Group"] = bx_plot_df["Age_Group"].replace({
    "AGE_0_19": "0–19",
    "AGE_20_64": "20–64",
    "AGE_65_PLUS": "65+"
})

fig_bx = px.box(
    bx_plot_df,
    x="Age_Group",
    y="Population",
    color="Age_Group",
    title="2025 Population Distribution per Age Group (Top-10 Communes, Excl. Genève)",
    labels={
        "Age_Group": "Age Group",
        "Population": "Population"
    }
)

fig_bx.update_layout(
    template="plotly_white",
    showlegend=False,
    title_font_size=20,
    )

fig_bx.update_yaxes(type="log", title_font_size=18, tickfont_size=16)
fig_bx.update_xaxes(title_font_size=18, tickfont_size=16)

fig_bx.show()

In [ ]:
# POPULATION PERCENTAGES
top10_pct = top10.copy()

top10_pct[age_cols] = (top10_pct[age_cols].div(top10_pct["TOTAL_POP"], axis=0)* 100)

 
# Stacked bar chart of age group percentage distributions for top-10 communes excluding Geneva
stkbr_plot_df = top10_pct.melt(
    id_vars="COMMUNE",
    value_vars=age_cols,
    var_name="Age_Group",
    value_name="Percentage"
)

# Cleaner labels
stkbr_plot_df["Age_Group"] = stkbr_plot_df["Age_Group"].replace({
    "AGE_0_19": "0-19",
    "AGE_20_64": "20-64",
    "AGE_65_PLUS": "65+"
})

# # STACKED BAR CHART
fig_stkbr = px.bar(
    stkbr_plot_df,
    x="COMMUNE",
    y="Percentage",
    color="Age_Group",
    title="Age Group Percentage Distribution (Top-10 Communes)",
    labels={
        "COMMUNE": "Commune",
        "Percentage": "Percentage (%)",
        "Age_Group": "Age Group"
    },
    barmode="stack"
)

# Optional styling
fig_stkbr.update_layout(
    xaxis_tickangle=-45,
    yaxis_title="Percentage (%)",
    legend_title="Age Group",
    template="plotly_white",
    title_font_size=22,
    legend_font_size=14
)

fig_stkbr.update_yaxes(title_font_size=20, tickfont_size=16)
fig_stkbr.update_xaxes(title_font_size=20, tickfont_size=16)

fig_stkbr.show()

In [ ]:
'''
GENDER ANALYSIS
'''

In [ ]:

# Scatter plot for the sex distribution of the top-10 communes excluding Geneva

fig_sct = px.scatter(
    top10,
    x="POP_HOMMES",
    y="POP_FEMMES",
    text="COMMUNE",
    size="TOTAL_POP",
    title="Male vs Female Population (Top-10 Communes)",
    labels={
        "POP_HOMMES": "Male Population",
        "POP_FEMMES": "Female Population"
    }
)


# Optional reference line y=x
max_val = max(
    top10["POP_HOMMES"].max(),
    top10["POP_FEMMES"].max()
)

fig_sct.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=max_val,
    y1=max_val,
    line=dict(dash="dash")
)

# default position for all points
positions = ["top center"] * len(top10)

# change only for Grand-Saconnex to avoid label overlap
positions = [
    "bottom center" if c == "Grand-Saconnex" or c == "Versoix"else "top center"
    for c in top10["COMMUNE"]
]

fig_sct.update_traces(
    textposition=positions,
    textfont_size=9
)

fig_sct.update_layout(template="plotly_white", title_font_size=22)

fig_sct.update_xaxes(range=[5000, None], title_font_size=20, tickfont_size=16)
fig_sct.update_yaxes(range=[5000, None], title_font_size=20, tickfont_size=16)

fig_sct.show()